# The X³Z³ Floquet code

The X³Z³ code of Setiawan & McLauchlan, *Tailoring Dynamical Codes for Biased Noise* (arXiv:2411.04974), on every
readout scheme here: the ancilla pairs and methods A, B and C. It is the CSS Floquet code of `floquet.py` with a
Hadamard on the data qubits of every other column of the brick wall (the paper's grey strips, Fig. 1b). Each
plaquette has three corners in each of its two columns, so its X⁶ / Z⁶ operators become **X³Z³** and **Z³X³**,
hence the name.

Why bother: in the CSS code a Z error only trips the X-type detectors, and under dephasing noise half of the
detectors are of little use. In X³Z³ a physical Z on a shaded qubit is an X in the CSS frame, so Z errors trip
both kinds, and the syndromes of Z errors pair up along each column (the paper's stabiliser-product symmetry):
the decoder has a much easier job when the noise is biased, and the threshold rises with the bias η instead of
falling.

In code it is one switch, `code="x3z3"`, taken by `floquet.memory_circuit` and every scheme's `round_circuit`:

* `floquet.shaded(q)`: the Hadamard-rotated data qubits, column x = 3 mod 4;
* `floquet.check_paulis(pauli, data, code)`: the Pauli each data qubit of a check gets, X ↔ Z on shaded ones, so
  an edge reads XX or ZZ inside a column and **XZ / ZX** across;
* `memory_circuit` resets and reads out the shaded qubits in the other basis. Records, detectors and the
  observable are the CSS code's: only which physical errors flip them changes.

In [ ]:
%load_ext autoreload
%autoreload 2

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import stim
from IPython.display import SVG, Image, display

import corner_readout
import floquet
import method_a
import method_b
import method_c
import pairs
import plots
from decode import SCHEMES
from drawing import COLORS
from floquet import (CODES, QUBIT_CORNERS, check_paulis, edge_list, hex_centers, memory_circuit, period, qubits,
                     shaded, step_colour, torus)
from memory import logical_errors
from noise import NOISE_MODELS, add_noise

In [ ]:
DISTANCE = 1  # 1 is 12 data qubits; 2 is 48
SCHEME = "method_b"  # whose steps to draw: pairs, method_a, method_b or method_c
NUMBERS = True  # label every qubit: Di data qubit i, Ak / Sk / Rk ancillas
from floquet import COLOUR_NAMES as names  # names[c]: colour c as drawn

## The shaded columns

Brick-wall coordinates, one torus period: data qubits sit in columns x = 1, 3, 5, ..., and the shaded ones
(x = 3 mod 4, grey strips) carry the Hadamard. The torus is 4d wide, so the strips alternate all the way round.

In [ ]:
p = period(DISTANCE)
fig, ax = plt.subplots(figsize=(4 + 2 * DISTANCE, 5 + 3 * DISTANCE))
for x in range(3, int(p.real), 4):
    ax.axvspan(x - 0.6, x + 0.6, color="0.85", zorder=0)
for h, c in hex_centers(DISTANCE).items():
    for shift in (0, p.real, -p.real, p.imag * 1j, -p.imag * 1j):
        corners = [h + shift + q for q in QUBIT_CORNERS]
        ax.fill([q.real for q in corners], [q.imag for q in corners], color=COLORS[c], alpha=0.25, edgecolor="0.4")
data = qubits(DISTANCE)
for flag, label in ((False, "unshaded: CSS Paulis"), (True, "shaded: X <-> Z")):
    qs = [q for q in data if shaded(q) == flag]
    ax.scatter([q.real for q in qs], [q.imag for q in qs], s=70, zorder=3, edgecolor="#333",
               facecolor="white" if flag else "#333", label=f"{label} ({len(qs)})")
for i, q in enumerate(data):
    ax.annotate(f"D{i}", (q.real, q.imag), xytext=(5, 3), textcoords="offset points", fontsize=7)
ax.set(xlim=(0, p.real), ylim=(p.imag, 0), aspect="equal", title=f"d = {DISTANCE}: X3Z3 data qubits")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
plt.show()

## What each step measures

The CSS schedule `rX gZ bX rZ gX bZ` stays; each check's Paulis come from `check_paulis`. Every step mixes
both kinds of check: vertical edges stay within one column (XX or ZZ), horizontal ones cross from one column to
the next (XZ or ZX).

In [ ]:
for r in range(6):
    checks = [check_paulis("XZ"[r % 2], e.data, "x3z3") for e in edge_list(DISTANCE) if e.colour == step_colour(r)]
    print(f"step {r}: CSS {'XZ'[r % 2] * 2} on {names[step_colour(r)]:5} edges -> X3Z3",
          dict(sorted(Counter(checks).items())))

The plaquettes the checks multiply to: one step's three checks around a plaquette of another colour. In the CSS
code that is X⁶ or Z⁶; here each plaquette is half X, half Z.

In [ ]:
corners = lambda h: [torus(h + q, DISTANCE) for q in QUBIT_CORNERS]
data_index = {q: i for i, q in enumerate(qubits(DISTANCE))}
for r in (0, 1):
    colour = step_colour(r)
    for h, c in list(hex_centers(DISTANCE).items())[:3]:
        if c == colour:
            continue
        product = stim.PauliString(len(data_index))
        for e in edge_list(DISTANCE):
            if e.colour == colour and set(e.data) <= set(corners(h)):
                check = stim.PauliString(len(data_index))
                for q, pauli in zip(e.data, check_paulis("XZ"[r % 2], e.data, "x3z3")):
                    check[data_index[q]] = pauli
                product *= check
        around = "".join("_XYZ"[product[data_index[q]]] for q in corners(h))
        print(f"step {r}: {names[c]:5} plaquette at {h}, corners clockwise from the right: {around}")

## The mixed check in each scheme

A ZZ check copies both data's Z onto a |0⟩ syndrome with CNOTs; an XX check starts the syndrome in |+⟩ and has it
control an X on each data qubit, then turns its X into the Z the spin-blockade readout (MZZ) sees. A mixed **XZ**
check is the XX one with a CZ for the Z side: CZ kicks Z of the data qubit back onto the syndrome's X. Each scheme
takes a two-letter Pauli, first letter for its first data qubit. Below, the XZ check on data qubits 0 and 1 in
each scheme (`pairs`: syndrome 2, reference 3; method A: syndrome 2, reference 3; methods B and C: syndrome a = 2,
s = 3, o = 4).

In [ ]:
def show(name, layers):
    print(name)
    for k, layer in enumerate(layers):
        print(f"    layer {k}: " + "  ".join(f"{g} {' '.join(map(str, t))}" for g, t in layer))

show("pairs", pairs.parity_check("XZ", 0, 1, 2, 3))
show("method A", method_a.corner_check("XZ", 0, 1, 2, 3))
show("method B, top-left corner", method_b.layers("XZ", 0, 1, 2, 3, 4, 4)[0])
show("method C", method_c.layers("XZ", 0, 1, 2, 3)[0])

A ZZ check has no H, so it is one layer shorter than XZ: in a step with both, `floquet.aligned` pads the shorter
checks with empty layers at the front, so every readout lands in the same tick and the records stay in
`edge_list` order. In the CSS code every check of a step has the same length and nothing changes; the CSS
circuits are identical to before X3Z3 was added.

## The six steps on hexagons

`SCHEME`'s X3Z3 circuit for each step, one panel per tick, plaquettes coloured underneath (as `make drawings`
draws the CSS ones). The gates are the CSS step's except on the mixed checks, where a CZ stands in for the
controlled-X on the Z side; steps that mix ZZ and XZ checks start the ZZ ones a tick later (`aligned`), so every
readout is in the same tick.

In [ ]:
def step_svg(scheme: str, r: int) -> str:
    if scheme == "pairs":
        return plots.round_svg(DISTANCE, r, True, NUMBERS, code="x3z3")
    if scheme == "method_a":
        return method_a.round_svg(DISTANCE, r, True, NUMBERS, code="x3z3")
    layers = {"method_b": method_b.layers, "method_c": method_c.layers}[scheme]
    return corner_readout.round_svg(DISTANCE, r, layers, True, NUMBERS, away=scheme == "method_c", code="x3z3")


for r in range(6):
    kinds = Counter(check_paulis("XZ"[r % 2], e.data, "x3z3") for e in edge_list(DISTANCE)
                    if e.colour == step_colour(r))
    print(f"step {r}: {names[step_colour(r)]} edges,", ", ".join(f"{n} {k}" for k, n in sorted(kinds.items())))
    display(SVG(step_svg(SCHEME, r)))

## What is checked

For both codes, each scheme's `check` asserts: record j holds check j's parity with the Paulis of
`check_paulis` (a stim flow), every data qubit is back in place, every two-qubit gate is between neighbours,
and the memory experiment's detectors are deterministic. Then both memory experiments of every scheme build a
detector error model without random detectors.

In [ ]:
method_a.check(DISTANCE)
corner_readout.check(DISTANCE, method_b.layers)
corner_readout.check(DISTANCE, method_c.layers)
for name, step in SCHEMES.items():
    for code in CODES:
        for basis in "ZX":
            memory_circuit(DISTANCE, 2, step, basis, code).detector_error_model(allow_gauge_detectors=False)
print(f"d = {DISTANCE}: every scheme checks out in both codes")

## CSS vs X3Z3 under bias

One point per scheme and code, at low and high bias: the Z and X memories at d = 3, p below threshold, each
decoded with its exact noise model (`memory.logical_errors`), combined as the paper's Eq. (9),
p_L = 1 − (1 − p_Z)(1 − p_X). Near depolarising (η = 1) the two codes are close; strongly dephasing (η = 1000)
X3Z3 is far ahead. A minute or two.

In [ ]:
D, P, SHOTS, NOISE = 3, 1e-3, 5_000, "spin"
for eta in (1, 1000):
    for name, step in SCHEMES.items():
        line = []
        for code in CODES:
            ok = 1.0
            for basis in "ZX":
                noisy = add_noise(memory_circuit(D, D, step, basis, code), NOISE_MODELS[NOISE](P, eta))
                errors, shots = logical_errors(noisy, SHOTS)
                ok *= 1 - errors / shots
            line.append(f"{code} {1 - ok:.2e}")
        print(f"η = {eta:<5} {name:9}", "   ".join(line))

## The full comparison

`make x3z3` sweeps every scheme, both codes, a range of η and p (`src/single_ancilla/x3z3.py`), and draws the
paper's figures: threshold and p_L vs η (Figs. 6, 7), and per η the LER vs p and LER vs physical qubits. The
threshold surface and threshold vs η_G, η_T of IBM's QEC-with-spin-qubits, for X3Z3: `make thresholds-x3z3`
(`thresholds-x3z3-medium` with more shots). If `make x3z3` has run, its figures:

In [ ]:
results = Path("../../results/single_ancilla/x3z3")
runs = sorted(results.glob("*_bias.png"))
if not runs:
    print("no results yet: make x3z3")
for bias in runs:
    run = str(bias)[:-len("_bias.png")]
    print(Path(run).name)
    display(Image(str(bias), width=900))
    for eta in ("1", "1000"):
        for kind in ("ler", "qubits"):
            png = Path(f"{run}_{kind}_eta{eta}.png")
            if png.exists():
                display(Image(str(png), width=900))